# Notebook 3a – Phòng thủ bằng Adversarial Training (Fast-AT, PGD-AT)

**Đề tài 2.21 – Nghiên cứu và phòng thủ trước tấn công đối kháng lên mô hình phân loại ảnh**

Người phụ trách: **Nguyễn Phương Đông** (phòng thủ)

Huấn luyện ResNet-18 **từ đầu** trên ảnh đối kháng, thay vì ảnh sạch:

- **Fast-AT** (Wong et al., 2020, *Fast is better than free*): mỗi batch sinh nhiễu bằng **1 bước FGSM có khởi tạo ngẫu nhiên** (α = 10/255, ε = 8/255), lịch học OneCycle. Rẻ gần bằng huấn luyện thường (~2 lần forward/backward mỗi batch).
- **PGD-AT** (Madry et al., 2018): mỗi batch sinh nhiễu bằng **PGD 7 bước** (α = 2/255). Bền vững hơn nhưng chậm hơn ~8 lần.

Chọn phương pháp bằng biến `MODE` ở mục 1. Sau mỗi epoch đo clean accuracy và PGD-10 accuracy (ε = 8/255) trên 1000 ảnh đánh giá cố định; mô hình tốt nhất (theo PGD accuracy) được lưu thành `adv_fast_resnet18.pth` hoặc `adv_pgd_resnet18.pth`.

> Giao diện dùng chung với phần tấn công: xem `docs/INTERFACE.md`.

## 0. Setup môi trường

- **Colab**: mount Google Drive (checkpoint lưu ở `MyDrive/AdvML_Project/checkpoints` để không mất khi runtime bị ngắt), clone repo vào `/content` nếu chưa có (có rồi thì `git pull`), rồi thêm repo vào `sys.path`.
- **Local**: thêm thư mục gốc repo (thư mục cha của `notebooks/`) vào `sys.path`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Eastern2k4/adversarial-attack-defense.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/AdvML_Project", exist_ok=True)

    REPO_DIR = Path("/content/adversarial-attack-defense")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    cwd = Path.cwd().resolve()
    REPO_DIR = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

### Copy dữ liệu CIFAR-10 từ Drive (nếu có)

Nếu đã để sẵn `cifar-10-python.tar.gz` trong `MyDrive/AdvML_Project/`, copy về `data/` để khỏi tải lại (torchvision sẽ tự giải nén). Không có file thì bỏ qua, dữ liệu sẽ được tải tự động.

In [ ]:
import shutil

DRIVE_DATA = Path("/content/drive/MyDrive/AdvML_Project/cifar-10-python.tar.gz")
LOCAL_DATA = Path("data") / DRIVE_DATA.name
if DRIVE_DATA.exists() and not LOCAL_DATA.exists():
    LOCAL_DATA.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(DRIVE_DATA, LOCAL_DATA)
    print("Đã copy", DRIVE_DATA, "->", LOCAL_DATA)
else:
    print("Bỏ qua copy dữ liệu (không có file trên Drive hoặc đã có sẵn).")

In [ ]:
import json
import time
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from src import config
from src.attacks import fgsm, pgd
from src.config import get_device
from src.data import get_loaders, get_eval_subset
from src.defenses import train_epoch_fast_at, train_epoch_pgd_at
from src.models import build_model
from src.utils import set_seed, evaluate, accuracy_under_attack, save_checkpoint, load_checkpoint

print("Checkpoint dir:", config.CHECKPOINT_DIR)

## 1. Chọn phương pháp, nạp dữ liệu

- `MODE = "fast"`: Fast-AT (khoảng 2 phút/epoch trên T4, ~1 giờ cho 30 epoch).
- `MODE = "pgd"`: PGD-AT 7 bước (khoảng 8–10 phút/epoch trên T4, ~4–5 giờ cho 30 epoch – có thể cần chạy qua nhiều phiên Colab, notebook tự resume).

Muốn có cả hai mô hình thì chạy notebook này **2 lần**, mỗi lần một `MODE`.

In [ ]:
MODE = "fast"  # "fast" hoặc "pgd"
assert MODE in ("fast", "pgd"), "MODE phải là 'fast' hoặc 'pgd'"

device = get_device()
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

set_seed(config.SEED)
train_loader, test_loader = get_loaders(batch_size=config.BATCH_SIZE)
x_eval, y_eval = get_eval_subset(as_tensors=True)
print(f"Train: {len(train_loader.dataset)} ảnh | Test: {len(test_loader.dataset)} ảnh | Eval subset: {len(x_eval)} ảnh")


def eps_key(eps):
    """8/255 -> "8/255" (khóa eps trong file kết quả, xem docs/INTERFACE.md)."""
    return f"{round(eps * 255)}/255"


# PGD-10, eps = 8/255: dùng để theo dõi độ bền vững sau mỗi epoch
pgd10 = partial(pgd, eps=config.AT_EPS, alpha=config.PGD_ALPHA, steps=config.PGD_STEPS)

## 2. Huấn luyện

- Mô hình khởi tạo **từ đầu** bằng `build_model()`; SGD momentum 0.9, weight decay 5e-4.
- **Fast-AT**: `OneCycleLR` (max_lr = `AT_LR_MAX`), scheduler step theo **từng batch**, giữ momentum cố định.
- **PGD-AT**: lr = `config.LR`, `CosineAnnealingLR` step theo epoch (giống baseline).
- Sau mỗi epoch lưu `adv_<mode>_last.pth` (model + optimizer + scheduler + lịch sử) → Colab bị ngắt thì chạy lại notebook, cell dưới **tự resume**.
- Lưu mô hình có **PGD accuracy cao nhất** (không phải clean accuracy) thành `config.ADV_CKPT_FAST` / `config.ADV_CKPT_PGD` — đây cũng là cách *early stopping* chống *catastrophic overfitting* của Fast-AT.

In [ ]:
model = build_model().to(device)
if MODE == "fast":
    optimizer = torch.optim.SGD(model.parameters(), lr=config.AT_LR_MAX,
                                momentum=config.MOMENTUM, weight_decay=config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=config.AT_LR_MAX, epochs=config.AT_EPOCHS,
        steps_per_epoch=len(train_loader), cycle_momentum=False)
    BEST_CKPT = config.CHECKPOINT_DIR / config.ADV_CKPT_FAST
else:
    optimizer = torch.optim.SGD(model.parameters(), lr=config.LR,
                                momentum=config.MOMENTUM, weight_decay=config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.AT_EPOCHS)
    BEST_CKPT = config.CHECKPOINT_DIR / config.ADV_CKPT_PGD

LAST_CKPT = config.CHECKPOINT_DIR / f"adv_{MODE}_last.pth"
config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

start_epoch, best_pgd = 0, 0.0
history = {"train_loss": [], "train_acc": [], "clean_acc": [], "pgd_acc": [], "lr": [], "epoch_time_sec": []}
if LAST_CKPT.exists():
    ckpt = load_checkpoint(LAST_CKPT, model, optimizer, scheduler, map_location=device)
    start_epoch, best_pgd = ckpt["epoch"], ckpt["best_acc"]
    history = ckpt.get("history", history)
    print(f"[{MODE}] Resume từ epoch {start_epoch}/{config.AT_EPOCHS}, best PGD acc = {best_pgd:.2f}%")
else:
    print(f"[{MODE}] Không có checkpoint, huấn luyện từ đầu.")

In [ ]:
for epoch in range(start_epoch, config.AT_EPOCHS):
    lr = optimizer.param_groups[0]["lr"]
    desc = f"[{MODE}] Epoch {epoch + 1}/{config.AT_EPOCHS}"
    t0 = time.time()
    if MODE == "fast":
        train_loss, train_acc = train_epoch_fast_at(model, train_loader, optimizer, scheduler,
                                                    config.AT_EPS, config.FAST_AT_ALPHA, device, desc=desc)
    else:
        train_loss, train_acc = train_epoch_pgd_at(model, train_loader, optimizer, config.AT_EPS,
                                                   config.AT_PGD_ALPHA, config.AT_PGD_STEPS, device, desc=desc)
        scheduler.step()
    if device.type == "cuda":
        torch.cuda.synchronize()
    epoch_time = time.time() - t0

    clean_acc = accuracy_under_attack(model, x_eval, y_eval)
    pgd_acc = accuracy_under_attack(model, x_eval, y_eval, pgd10)

    for k, v in zip(history, (train_loss, train_acc, clean_acc, pgd_acc, lr, epoch_time)):
        history[k].append(v)

    if pgd_acc > best_pgd:
        best_pgd = pgd_acc
        torch.save(model.state_dict(), BEST_CKPT)
    save_checkpoint(LAST_CKPT, model, optimizer, scheduler,
                    epoch=epoch + 1, best_acc=best_pgd, history=history)

    print(f"Epoch {epoch + 1:2d}/{config.AT_EPOCHS} | lr {lr:.4f} | {epoch_time:.0f}s | "
          f"train(adv) loss {train_loss:.4f} acc {train_acc:.2f}% | "
          f"clean {clean_acc:.2f}% | PGD-10 {pgd_acc:.2f}% | best PGD {best_pgd:.2f}%")

    # Catastrophic overfitting: độ bền vững trước PGD sụp đổ trong khi clean acc vẫn tăng
    if pgd_acc < 5.0 and len(history["clean_acc"]) >= 2 and clean_acc > history["clean_acc"][-2]:
        print("  ⚠️ CẢNH BÁO: PGD acc < 5% trong khi clean acc vẫn tăng -> có dấu hiệu CATASTROPHIC OVERFITTING. "
              f"Mô hình tốt nhất (PGD acc {best_pgd:.2f}%) vẫn được giữ ở {BEST_CKPT.name}.")

print(f"Hoàn tất. Best PGD-10 accuracy (eps 8/255): {best_pgd:.2f}%")

## 3. Biểu đồ huấn luyện

Train accuracy là accuracy trên **ảnh đối kháng** sinh lúc huấn luyện. Clean / PGD-10 accuracy đo trên 1000 ảnh đánh giá. Nếu đường PGD-10 tụt về ~0 trong khi clean tăng vọt → catastrophic overfitting.

In [ ]:
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
epochs = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))
ax1.plot(epochs, history["train_loss"])
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Train loss (ảnh đối kháng)"); ax1.grid(alpha=.3)
ax2.plot(epochs, history["train_acc"], label="train (adv)")
ax2.plot(epochs, history["clean_acc"], label="clean (eval)")
ax2.plot(epochs, history["pgd_acc"], label="PGD-10 8/255 (eval)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)"); ax2.set_title("Accuracy"); ax2.legend(); ax2.grid(alpha=.3)
ax3.bar(epochs, history["epoch_time_sec"])
ax3.set_xlabel("Epoch"); ax3.set_ylabel("Giây"); ax3.set_title("Thời gian huấn luyện mỗi epoch"); ax3.grid(alpha=.3)
plt.suptitle(f"Adversarial training – {MODE}")
plt.tight_layout()
plt.savefig(config.RESULTS_DIR / f"adv_{MODE}_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Đánh giá cuối

Nạp mô hình tốt nhất và đánh giá:

- **(a)** clean accuracy trên **toàn bộ** 10.000 ảnh test;
- **(b)** FGSM và PGD-10 (α = `config.PGD_ALPHA`) ở mọi ε trong `config.EPSILONS`, trên 1000 ảnh đánh giá;
- **(c)** đánh giá chéo: **PGD-20** ở ε = 8/255 (mạnh hơn tấn công dùng lúc huấn luyện). Với `MODE = "fast"`, mô hình được huấn luyện bằng FGSM nên **PGD là phương pháp tấn công khác** với lúc huấn luyện → kiểm tra mô hình có bền vững thật hay chỉ "học thuộc" FGSM. Với `MODE = "pgd"`, PGD-20 nhiều bước hơn PGD-7 lúc huấn luyện.

In [ ]:
def evaluate_attacks(model, name):
    """Đánh giá đầy đủ theo cấu trúc file kết quả (docs/INTERFACE.md, điểm 5)."""
    set_seed(config.SEED)  # cố định random start của PGD để so sánh công bằng giữa các mô hình
    res = {
        "model": name,
        "clean_acc": evaluate(model, test_loader, device),                     # (a) toàn bộ test set
        "clean_acc_eval_subset": accuracy_under_attack(model, x_eval, y_eval),
        "fgsm": {}, "pgd": {},
    }
    for eps in config.EPSILONS:                                                 # (b)
        res["fgsm"][eps_key(eps)] = accuracy_under_attack(
            model, x_eval, y_eval, partial(fgsm, eps=eps))
        res["pgd"][eps_key(eps)] = accuracy_under_attack(
            model, x_eval, y_eval, partial(pgd, eps=eps, alpha=config.PGD_ALPHA, steps=config.PGD_STEPS))
    res["pgd20"] = {eps_key(config.AT_EPS): accuracy_under_attack(              # (c)
        model, x_eval, y_eval, partial(pgd, eps=config.AT_EPS, alpha=config.PGD_ALPHA, steps=20))}
    return res


def print_result(res):
    print(f"== {res['model']} ==")
    print(f"  clean (test set): {res['clean_acc']:.2f}% | clean (eval subset): {res['clean_acc_eval_subset']:.2f}%")
    for k in res["fgsm"]:
        print(f"  eps {k:>6s}: FGSM {res['fgsm'][k]:6.2f}% | PGD-{config.PGD_STEPS} {res['pgd'][k]:6.2f}%")
    for k, v in res["pgd20"].items():
        print(f"  PGD-20 eps {k}: {v:.2f}%")


adv_model = build_model().to(device)
load_checkpoint(BEST_CKPT, adv_model, map_location=device)
adv_res = evaluate_attacks(adv_model, BEST_CKPT.stem)
print_result(adv_res)

## 5. So sánh với baseline

Nếu đã có `baseline_resnet18.pth` (Notebook 1), đánh giá baseline với **cùng bộ tấn công** và vẽ accuracy theo ε: nét liền = FGSM, nét đứt = PGD.

In [ ]:
BASELINE_PATH = config.CHECKPOINT_DIR / config.BASELINE_CKPT
base_res = None
if BASELINE_PATH.exists():
    base_model = build_model().to(device)
    load_checkpoint(BASELINE_PATH, base_model, map_location=device)
    base_res = evaluate_attacks(base_model, BASELINE_PATH.stem)
    print_result(base_res)
else:
    print(f"Chưa có {BASELINE_PATH} -> bỏ qua so sánh với baseline (chạy Notebook 1 trước).")

eps_x = [0] + [round(e * 255) for e in config.EPSILONS]
fig, ax = plt.subplots(figsize=(8, 5))
models_to_plot = [(adv_res, f"AT-{MODE}", "tab:blue")]
if base_res is not None:
    models_to_plot.append((base_res, "Baseline", "tab:red"))
for res, label, color in models_to_plot:
    clean = res["clean_acc_eval_subset"]
    ax.plot(eps_x, [clean] + list(res["fgsm"].values()), "-o", color=color, label=f"{label} – FGSM")
    ax.plot(eps_x, [clean] + list(res["pgd"].values()), "--s", color=color, label=f"{label} – PGD-{config.PGD_STEPS}")
ax.set_xticks(eps_x)
ax.set_xticklabels(["0"] + [f"{e}/255" for e in eps_x[1:]])
ax.set_xlabel("ε (L∞)"); ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 100)
ax.set_title(f"Accuracy theo ε – baseline vs adversarial training ({MODE})")
ax.grid(alpha=.3); ax.legend()
plt.tight_layout()
plt.savefig(config.RESULTS_DIR / f"adv_{MODE}_vs_baseline_eps.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Lưu kết quả

`results/adv_training_metrics.json` theo cấu trúc chung (`model`, `clean_acc`, `fgsm`, `pgd`) cộng thêm `mode`, `pgd20`, `train_time_per_epoch_sec`, `history`, `hyperparameters` và kết quả `baseline` (nếu có).
Vì chạy 2 `MODE` sẽ ghi đè file này, một bản sao riêng cho từng mode được lưu thêm ở `results/adv_training_<mode>_metrics.json`.

In [ ]:
metrics = {
    **adv_res,
    "mode": MODE,
    "train_time_per_epoch_sec": float(np.mean(history["epoch_time_sec"])) if history["epoch_time_sec"] else None,
    "best_epoch": int(np.argmax(history["pgd_acc"])) + 1 if history["pgd_acc"] else None,
    "hyperparameters": {
        "epochs": config.AT_EPOCHS, "eps": eps_key(config.AT_EPS), "batch_size": config.BATCH_SIZE,
        "momentum": config.MOMENTUM, "weight_decay": config.WEIGHT_DECAY,
        **({"alpha": f"{config.FAST_AT_ALPHA * 255:g}/255", "max_lr": config.AT_LR_MAX, "scheduler": "OneCycleLR"}
           if MODE == "fast" else
           {"alpha": eps_key(config.AT_PGD_ALPHA), "steps": config.AT_PGD_STEPS, "lr": config.LR,
            "scheduler": "CosineAnnealingLR"}),
    },
    "baseline": base_res,
    "history": history,
}
for name in ("adv_training_metrics.json", f"adv_training_{MODE}_metrics.json"):
    with open(config.RESULTS_DIR / name, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print("Đã lưu", config.RESULTS_DIR / name)

In [ ]:
# Trên Colab: sao lưu thư mục results/ lên Drive để không mất khi runtime kết thúc.
if IN_COLAB:
    dst = Path("/content/drive/MyDrive/AdvML_Project/results")
    shutil.copytree(config.RESULTS_DIR, dst, dirs_exist_ok=True)
    print("Đã sao lưu results/ lên", dst)